# IND320 – Project work, part 1: Dashboard basics
**Course:** IND320 Data to decision (NMBU, autumn 2026)
**Student:** Zarar Zafar

## Links
- **GitHub repository:** [https://github.com/zararzafar/zarar](https://github.com/zararzafar/zarar)
- **Streamlit app:** [https://zarar-ind320.streamlit.app](https://zarar-ind320.streamlit.app)

## Contents
1. AI usage
2. Work log
3. Reading the data
4. Renaming the headers
5. Inspecting the data
6. Plotting each column separately
7. Plotting all columns together

## 1. AI usage
I used Claude (Anthropic) as an AI assistant in this part of the project. It helped me find the
`reservoirs.csv` file in the course repository, set up the GitHub repository, write the first version of
the code for this notebook and for the Streamlit app, and suggest how to plot columns with very
different scales together. I went through the code, ran it, checked the plots against the raw data
and adjusted the comments so that I understand every step. All decisions about what to show and how
to interpret the data are my own responsibility.

## 2. Work log
*(300–500 words, describing both the Jupyter Notebook and the Streamlit work)*

I started this part of the project by setting up the tools that the rest of the course builds on.
I already had a GitHub account, so I created a new public repository called `zarar` with a README and
a Python `.gitignore`. I then logged in to share.streamlit.io with the same GitHub account, so that
Streamlit Community Cloud can read the repository and deploy the app directly from the `main` branch.

The data set for this part is `reservoirs.csv` from the course material (`D2Dbook/data` in the IND320
repository). It contains weekly reservoir statistics from NVE for Norway from 1995 until September 2026.
The first thing I noticed is that the file is in "long" format: every row is one week for one area,
and the areas are of three types – the whole country (`NO`), the five electricity price areas (`EL`,
NO1–NO5) and the water-course regions (`VASS`). The rows are also not sorted by date. To get one clean
time series I filtered on the national total (`omrType == "NO"`) and sorted by date, which gives 1653
weekly observations. The headers were in Norwegian, so I renamed them to short English names such as
`fill_rate`, `capacity_TWh` and `stored_energy_TWh`.

In the notebook I printed the data in a few complementary ways: the shape and data types, the first
rows, and summary statistics. After that I plotted each numeric column on its own axis, which shows the
strong yearly cycle in the fill rate: reservoirs are lowest in late April/early May before the snow
melt and highest in October. The capacity column turned out to be
constant (87.44 TWh) for the whole period, because NVE reports today's capacity for all years. Plotting all columns together was the most interesting
task, because the columns have different units (fractions, TWh and weekly changes). I solved this in
two ways: min–max scaling every column to the range 0–1, and a version with two y-axes where the
fractions are on the left and TWh on the right.

The Streamlit app has four pages, each in its own `.py` file. The home page uses the built-in
multipage sidebar for navigation. All pages load the data through one shared function decorated with
`@st.cache_data`, so the CSV is only read once. The second page shows a table with one row per
column of the data and a `LineChartColumn` with the values from the first month (January 1995). The
third page has a `selectbox` to choose one column or all columns, and a `select_slider` to choose a range
of months, with the first month as the default. The fourth page is a placeholder for later parts.

The main things I learned were how Streamlit reruns the whole script on every interaction (which is
why caching matters), how the multipage folder structure works, and that deploying early makes it
much easier to spot missing packages in `requirements.txt`. In part 2 the CSV file will be replaced by
data from MongoDB.

## 3. Reading the data
The file `reservoirs.csv` comes from the course repository (`khliland/IND320`, folder `D2Dbook/data`).
A copy is stored in the `data/` folder of my own repository so that both this notebook and the
Streamlit app use the same file.

In [ ]:
# Import the libraries used in this notebook
import pandas as pd                  # data handling
import matplotlib.pyplot as plt      # plotting
import matplotlib.dates as mdates    # nicer date axes

# Path to the CSV file (relative to the repository root)
DATA_PATH = "data/reservoirs.csv"

# Read the CSV file. The two date columns are parsed as dates directly.
raw = pd.read_csv(DATA_PATH, parse_dates=["dato_Id"])

# Quick check of the size of the raw file: (rows, columns)
print("Raw data shape:", raw.shape)

## 4. Renaming the headers
The original headers are Norwegian abbreviations from NVE. They are renamed to English names that
explain the content.

In [ ]:
# Mapping from the original Norwegian headers to English, self-explanatory names
column_names = {
    "dato_Id": "date",                                    # date of the weekly observation
    "omrType": "area_type",                               # NO = Norway, EL = price area, VASS = water course
    "omrnr": "area_number",                               # number of the area (0 = whole country)
    "iso_aar": "iso_year",                                # ISO year
    "iso_uke": "iso_week",                                # ISO week number
    "fyllingsgrad": "fill_rate",                          # share of capacity that is filled (0-1)
    "kapasitet_TWh": "capacity_TWh",                      # total reservoir capacity in TWh
    "fylling_TWh": "stored_energy_TWh",                   # energy stored in the reservoirs in TWh
    "neste_Publiseringsdato": "next_publication_date",    # when NVE publishes the next value
    "fyllingsgrad_forrige_uke": "fill_rate_previous_week",# fill rate one week earlier
    "endring_fyllingsgrad": "fill_rate_change",           # change in fill rate since previous week
}
reservoirs = raw.rename(columns=column_names)

# Show the new column names
print(list(reservoirs.columns))

## 5. Inspecting the data
The file is in *long* format: each row is one week for one area. The areas are the whole country
(`NO`), the five price areas (`EL`) and the water-course regions (`VASS`).

In [ ]:
# How many rows are there per area type and area number?
print(reservoirs.groupby(["area_type", "area_number"]).size().rename("rows").to_string())

To get one clean time series, the rest of the notebook (and the Streamlit app) uses the national
total for Norway (`area_type == "NO"`), sorted by date.

In [ ]:
# Keep only the national total and sort chronologically (the raw file is not sorted)
norway = (reservoirs[reservoirs["area_type"] == "NO"]
          .sort_values("date")
          .set_index("date"))

# The numeric measurement columns that will be plotted
value_columns = ["fill_rate", "capacity_TWh", "stored_energy_TWh",
                 "fill_rate_previous_week", "fill_rate_change"]

# Overview: number of rows, period covered and data types
print(f"Weekly observations: {len(norway)}")
print(f"Period: {norway.index.min().date()} to {norway.index.max().date()}")
print()
norway[value_columns].info()

In [ ]:
# The first rows of the measurement columns
norway[value_columns].head(10)

In [ ]:
# Summary statistics for the measurement columns (rounded for readability)
norway[value_columns].describe().round(3)

## 6. Plotting each column separately
Each measurement column is plotted on its own axis so that its own scale and pattern is visible.

In [ ]:
# Nice titles and y-axis labels for each column
labels = {
    "fill_rate": ("Fill rate", "Share of capacity (0-1)"),
    "capacity_TWh": ("Reservoir capacity", "TWh"),
    "stored_energy_TWh": ("Stored energy", "TWh"),
    "fill_rate_previous_week": ("Fill rate previous week", "Share of capacity (0-1)"),
    "fill_rate_change": ("Weekly change in fill rate", "Change (fraction points)"),
}

# One subplot per column, sharing the time axis
fig, axes = plt.subplots(len(value_columns), 1, figsize=(11, 13), sharex=True)
for ax, col in zip(axes, value_columns):
    title, ylabel = labels[col]
    ax.plot(norway.index, norway[col], linewidth=0.8, color="tab:blue")
    ax.set_title(title)
    ax.set_ylabel(ylabel)
    ax.grid(alpha=0.3)
axes[-1].set_xlabel("Date")
axes[-1].xaxis.set_major_locator(mdates.YearLocator(base=3))
fig.suptitle("Norwegian hydropower reservoirs, weekly (1995-2026)", fontsize=14)
fig.tight_layout()
plt.show()

**Observations:** the fill rate has a clear yearly cycle (lowest around week 16-18, highest around
week 40-42). The capacity is constant at 87.44 TWh for the whole period (NVE uses the current capacity for all years).
The weekly change is positive during snow melt and negative during winter.

## 7. Plotting all columns together
The columns have very different units and ranges (fractions around 0-1, capacity around 87 TWh,
weekly changes around ±0.05). Plotted directly on one axis, the fractions would look flat next to the
TWh values. Two solutions are shown:

1. **Min–max scaling** – every column is scaled to 0–1, so the shapes can be compared on one axis.
2. **Two y-axes** – fractions on the left axis and TWh on the right axis, keeping the real units.

To make the curves readable, the last three years are shown.

In [ ]:
# Min-max scale each column to the range 0-1: (x - min) / (max - min)
# A constant column (here: capacity) has max == min, which would divide by zero,
# so such columns are drawn as a flat line at 0.5 instead.
def min_max_scale(frame):
    value_range = frame.max() - frame.min()
    scaled = (frame - frame.min()) / value_range.replace(0, 1)
    scaled.loc[:, value_range == 0] = 0.5
    return scaled

scaled = min_max_scale(norway[value_columns])

# Show only the last three years so the individual curves can be told apart
recent = scaled.loc[scaled.index >= scaled.index.max() - pd.DateOffset(years=3)]

fig, ax = plt.subplots(figsize=(11, 5))
for col in value_columns:
    ax.plot(recent.index, recent[col], label=labels[col][0], linewidth=1.3)
ax.set_title("All columns together, min-max scaled (last three years)")
ax.set_xlabel("Date")
ax.set_ylabel("Scaled value (0 = column minimum, 1 = column maximum)")
ax.grid(alpha=0.3)
ax.legend(loc="upper left", fontsize=9)
fig.tight_layout()
plt.show()

In [ ]:
# Alternative: keep the original units using two y-axes
recent_raw = norway.loc[norway.index >= norway.index.max() - pd.DateOffset(years=3), value_columns]

fig, ax_left = plt.subplots(figsize=(11, 5))
ax_right = ax_left.twinx()   # second y-axis sharing the same x-axis

# Fractions on the left axis
for col in ["fill_rate", "fill_rate_previous_week", "fill_rate_change"]:
    ax_left.plot(recent_raw.index, recent_raw[col], label=labels[col][0], linewidth=1.3)
# Energy in TWh on the right axis (dashed lines to separate them visually)
for col, color in [("capacity_TWh", "black"), ("stored_energy_TWh", "tab:purple")]:
    ax_right.plot(recent_raw.index, recent_raw[col], label=labels[col][0] + " (TWh)",
                  linestyle="--", color=color, linewidth=1.3)

ax_left.set_xlabel("Date")
ax_left.set_ylabel("Fill rate / change (fraction)")
ax_right.set_ylabel("Energy (TWh)")
ax_left.set_title("All columns together with two y-axes (last three years)")
ax_left.grid(alpha=0.3)

# Combine the legends from both axes into one box
lines = ax_left.get_lines() + ax_right.get_lines()
ax_left.legend(lines, [l.get_label() for l in lines], loc="lower left", fontsize=9)
fig.tight_layout()
plt.show()

## Next steps
In part 2 the local CSV file will be replaced by data stored in MongoDB, both in this notebook and in
the Streamlit app.